# 0. Configuration

In [17]:
# data load
from pathlib import Path # help with folder and file path 

# paths
import os 
# dataframe
import pandas as pd
import numpy as np
from functools import reduce

# data visualization
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Cross Validation
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Model
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import roc_auc_score, precision_recall_curve


In [18]:
# build paths

base_dir = os.getcwd()

# join parts of the path

def get_pd(folder,name):
    path = os.path.join('datasets',folder,name)
    return path

# 1. Load Data

In [19]:
X_train_path = get_pd('train','X_train.csv')
X_train = pd.read_csv(X_train_path)
X_train = X_train.iloc[:,1:] # drop the first column which is just index
X_train

,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,...,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions
0,Missing,1.0,42.925145,140.053691,386.0,17.0,11.0,1.0,10.0,34.0,...,5.0,-999.0,4.0,5.0,1.0,23.98,3.0,-999.0,-999.0,2.505361
1,Missing,1.0,444.602262,134.242133,386.0,17.0,11.0,1.0,15.0,30.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,4.483389
2,False,2.0,314.989689,4.757439,386.0,17.0,12.0,3.0,11.0,33.0,...,10.0,-999.0,5.0,7.0,2.0,0.00,4.0,-999.0,-999.0,12.503344
3,False,4.0,314.989689,22.485073,1100.0,17.0,9.0,1.0,11.0,33.0,...,2.0,-999.0,3.0,2.0,1.0,0.00,3.0,-999.0,-999.0,19.816075
4,False,2.0,314.989689,633.680181,64.0,17.0,6.0,1.0,11.0,24.0,...,0.0,-999.0,1.0,0.0,1.0,66.46,1.0,-999.0,-999.0,2.481141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13183,Missing,3.0,6.373022,363.475348,386.0,17.0,9.0,1.0,12.0,34.0,...,0.0,-999.0,1.0,0.0,1.0,55.55,1.0,-999.0,-999.0,4.483389
13184,Missing,2.0,158.621297,2.193305,730.0,17.0,15.0,1.0,12.0,35.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,1.0,4.101334
13185,False,-999.0,-999.000000,7.180110,386.0,17.0,15.0,-999.0,13.0,7.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
13186,Missing,1.0,1511.193006,871.043785,386.0,17.0,15.0,1.0,12.0,5.0,...,1.0,-999.0,3.0,1.0,1.0,33.14,2.0,-999.0,-999.0,11.629628


In [20]:
y_train_path = get_pd('train','y_train.csv')
y_train = pd.read_csv(y_train_path)
y_train = y_train.iloc[:,1:] # drop the first column which is just index
y_train

,is_fraudulent
0,0
1,0
2,1
3,1
4,0
...,...
13183,0
13184,0
13185,0
13186,0


# 2. Feature Engineering

Feature engineering is part of the model development pipeline and can also be part of model selection when different feature sets are compared using validation results. In this project, feature transformations that learn from the data, such as encoding, imputation, scaling, and resampling, must be applied within the cross-validation process to **avoid data leakage**. This ensures that each validation fold remains unseen during training and gives a more reliable estimate of model performance.


## 2a. K-Fold Cross Validation

After splitting the data into training and test sets, I set up a cross-validation (CV) loop on the training data only. Cross-validation is a resampling method used during model development to evaluate how well a model is likely to generalize to unseen data.

In this project, I use Stratified K-Fold cross-validation, which repeatedly splits the training set into several folds while preserving the original fraud ratio (~6.5%) in each fold. For each iteration, the model is trained on `K-1` folds and validated on the remaining fold. This process is repeated until every fold has been used once as a validation set, and the performance is then averaged across all folds.

This step is especially important for an imbalanced fraud dataset because a single validation split may give an unstable estimate of model performance. Stratified cross-validation provides a more reliable assessment while maintaining the minority fraud class proportion in each fold.

Cross-validation is only applied to the training set, not the test set. The test set is kept untouched until the very end so it can provide a final unbiased evaluation of the selected model.


In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2b. Preprocessor

Before fitting the model on training set, preprocessing categorical values and standardizing numerical values are important. For categorical values, there are several methods, but the key is to keep in mind about the data's cardinality.

My training data has:
- latest_changed_password: 3 unique values
- latest_item_category: about 3,109 unique values
- latest_item_product_title: about 6,906 unique values


Best simple baseline: one-hot low-cardinality + frequency encoding high-cardinality

Best stronger experiment: one-hot low-cardinality + target encoding high-cardinality

Best model-driven option: try CatBoost with native categorical handling


In [ ]:
class FrequencyEncoder (BaseEstimator, TransformerMixin):
    def __init__(self, normalize = True):
        self.normalize = normalize

    

In [22]:
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X_train.select_dtypes(include=["float64","int64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers = [
        # handle unknown category
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features),
        ("num",StandardScaler(), numerical_features),
        
    ]
)


## 2c. Pipeline

In [23]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

## 2d. CV used on Pipeline 

In [24]:
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="average_precision"
)

/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was e

In [25]:
scores

array([0.66985826, 0.69998906, 0.68058829, 0.72831551, 0.6863205 ])

# 3. Feature Selection

# 4. Baseline Model: Train & Test

In [43]:
# create XGB model

xgb_clf = XGBClassifier(
    objective='binary:logistic',
    n_estimators=30,
    learning_rate=0.1,
    max_depth=10,
    random_state=42,
    enable_categorical=True,
    use_label_encoder=False,
    eval_metric='logloss'
)


# create pipeline to train model
model = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",xgb_clf)
])

# fit model
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_train)
y_proba = model.predict_proba(X_train)[:,1]


/Users/chohasong/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [20:11:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# 5. Baseline Model: Evalation

In [ ]:
# on train 
roc_auc_score(y_train,y_proba)

0.9994726550650781

# References

https://www.ibm.com/think/tutorials/gradient-boosting-classifier#179371088

https://xgboost.readthedocs.io/en/release_3.2.0/get_started.html


https://feature-engine.trainindata.com/en/1.8.x/user_guide/encoding/WoEEncoder.html
